# 使用LayoutLMv3进行自定义数据微调和推理

## 说明

LayoutLMv3模型能够在FUNSD上获得超过90%的F1分数。这得益于使用段位置嵌入，而不是基于单词的位置信息，灵感来自于[StructuralLM](https://arxiv.org/abs/2105.11210)。这意味着属于同一“段”（例如，一个地址）的单词会获得相同的边界框坐标，因此具有相同的二维位置嵌入。

大多数OCR引擎（如谷歌的Tesseract）能够识别段，如LayoutLMv3作者在[这个讨论](https://github.com/microsoft/unilm/issues/838)中所解释的那样。

对于FUNSD数据集，段是根据标签创建的，如[这里](https://huggingface.co/datasets/nielsr/funsd-layoutlmv3/blob/main/funsd-layoutlmv3.py#L140)所示。

始终建议使用段位置嵌入而不是基于单词的位置信息，因为这会显著提高性能。

## 训练提示

请注意，LayoutLMv3在训练/推理方面与LayoutLMv2是相同的，除了以下几点：

* 图像需要被调整大小和归一化，以使其成为形状为 `(batch_size, num_channels, height, width)`的 `pixel_values`。通道需要为RGB格式。这与LayoutLMv2不同，后者期望通道为BGR格式（由于其Detectron2视觉骨干），并在内部对图像进行了归一化。
* 文本的标记化基于RoBERTa，因此使用字节级的字节对编码（Byte-Pair-Encoding）。这与LayoutLMv2使用的BERT风格的WordPiece标记化不同。

因此，我创建了一个新的 `LayoutLMv3Processor`，它将 `LayoutLMv3ImageProcessor`（用于图像模态）和 `LayoutLMv3TokenizerFast`（用于文本模态）结合为一个。用法与其前身[`LayoutLMv2Processor`](https://huggingface.co/docs/transformers/model_doc/layoutlmv2#usage-layoutlmv2processor)相同。

## 环境设置
1. MindSpore 2.4.10
2. Mindnlp 0.4.1
3. Python 3.9



**使用华为云 ModelArts 作为AI平台**

在环境搭建部分，使用了AI gallery社区中相关mindnlp项目搭建mindnlp环境的代码。

### 环境配置

配置python3.9环境

In [ ]:
%%capture captured_output

# 安装python 3.9版本的kernel
!/home/ma-user/anaconda3/bin/conda create -n python-3.9.0 python=3.9.0 -y --override-channels --channel https://mirrors.tuna.tsinghua.edu.cn/anaconda/pkgs/main
!/home/ma-user/anaconda3/envs/python-3.9.0/bin/pip install ipykernel

In [ ]:
import json
import os

data = {
   "display_name": "python-3.9.0",
   "env": {
      "PATH": "/home/ma-user/anaconda3/envs/python-3.9.0/bin:/home/ma-user/anaconda3/envs/python-3.7.10/bin:/modelarts/authoring/notebook-conda/bin:/opt/conda/bin:/usr/local/nvidia/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/home/ma-user/modelarts/ma-cli/bin:/home/ma-user/modelarts/ma-cli/bin"
   },
   "language": "python",
   "argv": [
      "/home/ma-user/anaconda3/envs/python-3.9.0/bin/python",
      "-m",
      "ipykernel",
      "-f",
      "{connection_file}"
   ]
}

if not os.path.exists("/home/ma-user/anaconda3/share/jupyter/kernels/python-3.9.0/"):
    os.mkdir("/home/ma-user/anaconda3/share/jupyter/kernels/python-3.9.0/")

with open('/home/ma-user/anaconda3/share/jupyter/kernels/python-3.9.0/kernel.json', 'w') as f:
    json.dump(data, f, indent=4)

*注：以上代码执行完成后，需点击左上角或右上角将kernel更换为python-3.9.0*

2. 安装mindspore2.4.10，安装指南详见：[MindSpore安装](https://www.mindspore.cn/install/)

3. 安装MindNLP及相关依赖，MindNLP官方仓详见：[MindNLP](https://github.com/mindspore-lab/mindnlp)

创建完成后，稍等片刻，或刷新页面，点击右上角（或左上角）kernel选择python-3.9

安装python依赖

In [ ]:
!pip install https://ms-release.obs.cn-north-4.myhuaweicloud.com/2.4.10/MindSpore/unified/aarch64/mindspore-2.4.10-cp310-cp310-linux_aarch64.whl --trusted-host ms-release.obs.cn-north-4.myhuaweicloud.com -i https://pypi.tuna.tsinghua.edu.cn/simple
!pip install mindnlp==0.4.1

In [ ]:
!pip show mindspore
!pip show mindnlp

In [ ]:
!pip install seqeval

In [ ]:
import mindspore

mindspore.context.get_context('device_target')

## 加载数据集

接下来，我们从 🤗 [hub](https://huggingface.co/datasets/nielsr/funsd-layoutlmv3) 加载一个数据集。这个数据集是 [FUNSD](https://guillaumejaume.github.io/FUNSD/) 数据集，包含了一系列带注释的表单。


In [ ]:
!git clone https://github.com/wjy4399/funsd-layoutlmv3

In [ ]:
from mindnlp.dataset import load_dataset 

# this dataset uses the new Image feature :)
dataset = load_dataset("funsd-layoutlmv3")

In [ ]:
train_dataset = dataset["train"]
test_dataset = dataset["test"]

正如我们所见，数据集由2个拆分部分组成（"训练集"和"测试集"），每个样本都包含一个单词列表（"tokens"）及其对应的边界框（"bboxes"），而且这些单词都被标记了（"ner_tags"）。每个样本还包括原始图像（"image"）。

In [ ]:
example = next(train_dataset.create_dict_iterator())
print(example)

查看示例

In [ ]:
from PIL import Image
import io
image = example['image'].asnumpy()
image = Image.fromarray(image, 'RGB')
# 显示图片
image.show()

In [ ]:
words, boxes, ner_tags = example["tokens"], example["bboxes"], example["ner_tags"]
print(words)
print(boxes)
print(ner_tags)

## 准备数据集

接下来，我们为模型准备数据集。这可以使用`LayoutLMv3Processor`非常轻松地完成，它在内部将`LayoutLMv3FeatureExtractor`（用于图像模态）和`LayoutLMv3Tokenizer`（用于文本模态）包装在一起。

基本上，处理器在内部执行以下操作：
* 特征提取器用于将每个文档图像调整大小并归一化为`pixel_values`
* 分词器用于将单词、边界框和NER标签转换为标记级别的`input_ids`、`attention_mask`和`labels`。

处理器简单地返回一个包含所有这些键的字典。


In [ ]:
from mindnlp.transformers import AutoProcessor

# we'll use the Auto API here - it will load LayoutLMv3Processor behind the scenes,
# based on the checkpoint we provide from the hub
processor = AutoProcessor.from_pretrained("microsoft/layoutlmv3-base", apply_ocr=False)

我们首先创建id2label和label2id映射，这对推理很有用。请注意，LayoutLMv3ForTokenClassification（我们稍后将使用的模型）将简单地为每个标记输出一个特定类别的整数索引，因此我们仍然需要将其映射到实际的类别名称。

In [ ]:
# 2. 定义列名和标签映射
image_column_name = "image"
text_column_name = "tokens"
boxes_column_name = "bboxes"
label_column_name = "ner_tags"
# 定义标签映射
label_list = ['O', 'B-HEADER', 'I-HEADER', 'B-QUESTION', 'I-QUESTION', 'B-ANSWER', 'I-ANSWER']
id2label = {i: name for i, name in enumerate(label_list)}
label2id = {name: i for i, name in enumerate(label_list)}
num_labels = len(label_list)

In [ ]:
print(label_list)

In [ ]:
print(id2label)

接下来，我们将定义一个可以应用于整个数据集的函数。

In [ ]:
def prepare_examples(examples):
  images = examples[image_column_name]
  words = examples[text_column_name]
  boxes = examples[boxes_column_name]
  word_labels = examples[label_column_name]

  encoding = processor(images, words, boxes=boxes, word_labels=word_labels,
                       truncation=True, padding="max_length")

  return encoding

In [ ]:

class FUNSDDataset:
    """FUNSD数据集的自定义处理类"""

    def __init__(self, dataset, processor, max_length=512):
        """
        Args:
            dataset: MindNLP加载的数据集
            processor: LayoutLMv3的处理器
            max_length: 序列最大长度
        """
        self.dataset = dataset
        self.processor = processor
        self.max_length = max_length
        
        # 提取所有需要的数据
        self.images = []
        self.tokens = []
        self.boxes = []
        self.labels = []
        
        # 加载所有样本
        for item in self.dataset.create_dict_iterator():
            self.images.append(item[image_column_name].asnumpy())
            self.tokens.append(item[text_column_name].asnumpy().tolist())
            self.boxes.append(item[boxes_column_name].asnumpy().tolist())
            self.labels.append(item[label_column_name].asnumpy().tolist())
        
        # 预处理所有数据
        self.encoded_inputs_list = []
        self.__process_all_samples__()
    
    def __process_all_samples__(self):
        """预处理所有样本"""
        for idx in range(self.__len__()):
            # 获取单个样本
            image = self.images[idx]
            words = self.tokens[idx]
            boxes = self.boxes[idx]
            word_labels = self.labels[idx]
            
            # 确保数据长度一致
            assert len(words) == len(boxes) == len(word_labels)
            
            # 使用处理器预处理
            encoded_inputs = self.processor(
                image, 
                words, 
                boxes=boxes, 
                word_labels=word_labels,
                truncation=True, 
                padding="max_length", 
                max_length=self.max_length,
                return_tensors="np"
            )
            
            # 去除批处理维度
            for k, v in encoded_inputs.items():
                encoded_inputs[k] = v.squeeze()
            
            # 存储为元组，适配MindSpore的GeneratorDataset
            self.encoded_inputs_list.append((
                encoded_inputs["pixel_values"],
                encoded_inputs["input_ids"],
                encoded_inputs["attention_mask"],
                encoded_inputs["bbox"],
                encoded_inputs["labels"]
            ))

    def __len__(self):
        """返回数据集长度"""
        return len(self.images)

    def __getitem__(self, idx):
        """获取单个样本"""
        return self.encoded_inputs_list[idx]

In [ ]:
# 使用MindSpore的数据处理方式
from mindspore.dataset import GeneratorDataset

# 创建自定义数据集实例
train_funsd_dataset = FUNSDDataset(dataset["train"], processor)
eval_funsd_dataset = FUNSDDataset(dataset["test"], processor)

In [ ]:
# 定义列名
column_names = ["pixel_values", "input_ids", "attention_mask", "bbox", "labels"]

# 创建MindSpore数据集
def create_ms_dataset(custom_dataset, batch_size=2, shuffle=True):
    """创建MindSpore数据集"""
    ms_dataset = GeneratorDataset(
        source=custom_dataset,
        column_names=column_names,
        shuffle=shuffle
    )
    
    # 批处理
    ms_dataset = ms_dataset.batch(batch_size)
    
    return ms_dataset

In [ ]:
# 创建最终的数据集
train_dataset = create_ms_dataset(train_funsd_dataset)
eval_dataset = create_ms_dataset(eval_funsd_dataset)


让我们验证一下所有内容是否正确创建：

In [ ]:
# 展示处理后的样本
first_batch = next(train_dataset.create_dict_iterator())
print("训练样本数量:", len(train_funsd_dataset))
print("处理后的数据形状:")
for key, value in first_batch.items():
    print(f"  {key}: {value.shape}")


In [ ]:
# 解码第一个样本的input_ids
decoded_text = processor.tokenizer.decode(first_batch["input_ids"][0].asnumpy())
print("\n解码后的文本示例:\n", decoded_text[:200] + "...")

## 定义模型

接下来，我们定义模型：这是一个带有预训练权重的 Transformer 编码器，顶部有一个随机初始化的头用于标记分类。


In [ ]:
from mindnlp.transformers import LayoutLMv3ForTokenClassification

model = LayoutLMv3ForTokenClassification.from_pretrained(
        "microsoft/layoutlmv3-base",
        num_labels=num_labels,
        id2label=id2label,
        label2id=label2id
    )
    

In [ ]:
model

In [ ]:
print(type(model))

## 创建梯度计算函数

接下来，我们定义 `TrainingArguments`，它定义了与训练相关的所有超参数。请注意，有大量参数可以调整，更多信息请查看 [文档](https://huggingface.co/docs/transformers/main_classes/trainer#transformers.TrainingArguments)。


In [ ]:
from mindnlp.core.optim import AdamW
optimizer = AdamW(model.trainable_params(), lr=1e-5)

from mindnlp.core.autograd import value_and_grad

def forward_fn(batch):
    """
    前向传播函数
    
    Args:
        batch: 一个批次的数据，包含输入和标签
        
    Returns:
        loss: 模型计算的损失值
    """
    # 获取输入
    input_ids = batch['input_ids']
    bbox = batch['bbox']
    pixel_values = batch['pixel_values']  # LayoutLMv3使用pixel_values而不是image
    attention_mask = batch['attention_mask']
    labels = batch['labels']
    
    # token_type_ids在我们的数据集中可能不存在，所以不传入
    outputs = model(
        input_ids=input_ids,
        bbox=bbox,
        pixel_values=pixel_values,  # 使用pixel_values
        attention_mask=attention_mask,
        labels=labels
    ) 
    
    loss = outputs.loss
    return loss

# 创建梯度计算函数
grad_fn = value_and_grad(forward_fn, model.trainable_params(), attach_grads=True)

## 训练模型

In [ ]:
from tqdm import tqdm

# put the model in training mode
model.set_train(True)
global_step = 0
num_train_epochs = 10
epoch=0
# 计算每个epoch的步数
train_size = len(train_funsd_dataset)  # 数据集大小
batch_size = 2  # 批处理大小(默认值)
steps_per_epoch = (train_size + batch_size - 1) // batch_size  # 向上取整

for epoch in range(num_train_epochs):  
    print("Epoch:", epoch)
    for batch in tqdm(train_dataset.create_dict_iterator(), total=steps_per_epoch):
        optimizer.zero_grad()
        # forward, backward + optimize
        loss = grad_fn(batch)
        optimizer.step()


        # print loss every 100 steps
        if global_step % 1 == 0:
            print(f"Loss after {global_step} steps: {loss.item()}")

        global_step += 1

## 保存模型

In [ ]:
model.save_pretrained("./LayoutLMv3/Checkpoints20250423")

In [ ]:
processor.save_pretrained('./LayoutLMv3/processorPretrained')

In [ ]:
pretrained_model_path = "./LayoutLMv3/Checkpoints20250423"
model = LayoutLMv3ForTokenClassification.from_pretrained(pretrained_model_path)

评估结果

In [ ]:

# 验证过程
import numpy as np
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")
from seqeval.metrics import (
    classification_report,
    f1_score,
    precision_score,
    recall_score)

# 创建测试数据加载器
test_dataset = create_ms_dataset(eval_funsd_dataset, batch_size=2, shuffle=False)

preds_val = None
out_label_ids = None

# 设置模型为评估模式
model.set_train(False)
for batch in tqdm(test_dataset.create_dict_iterator(), desc="评估中"):
    input_ids = batch['input_ids']
    bbox = batch['bbox']
    pixel_values = batch['pixel_values']  # LayoutLMv3使用pixel_values而不是image
    attention_mask = batch['attention_mask']
    labels = batch['labels']

    # 前向传播
    outputs = model(
        input_ids=input_ids,
        bbox=bbox,
        pixel_values=pixel_values,  # 使用pixel_values
        attention_mask=attention_mask,
        labels=labels
    )

    if preds_val is None:
        preds_val = outputs.logits.asnumpy()
        out_label_ids = batch["labels"].asnumpy()
    else:
        preds_val = np.append(preds_val, outputs.logits.asnumpy(), axis=0)
        out_label_ids = np.append(out_label_ids, batch["labels"].asnumpy(), axis=0)

def results_test(preds, out_label_ids, labels):
    """
    计算评估指标
    
    Args:
        preds: 预测的logits (batch_size, seq_len, num_labels)
        out_label_ids: 真实标签 (batch_size, seq_len)
        labels: 标签列表
        
    Returns:
        results: 总体评估指标 (precision, recall, f1)
        class_report: 各类别的详细评估报告
    """
    preds = np.argmax(preds, axis=2)

    label_map = {i: label for i, label in enumerate(labels)}

    out_label_list = [[] for _ in range(out_label_ids.shape[0])]
    preds_list = [[] for _ in range(out_label_ids.shape[0])]

    for i in range(out_label_ids.shape[0]):
        for j in range(out_label_ids.shape[1]):
            if out_label_ids[i, j] != -100:
                out_label_list[i].append(label_map[out_label_ids[i][j]])
                preds_list[i].append(label_map[preds[i][j]])

    results = {
        "precision": precision_score(out_label_list, preds_list),
        "recall": recall_score(out_label_list, preds_list),
        "f1": f1_score(out_label_list, preds_list),
    }
    return results, classification_report(out_label_list, preds_list)

# 获取标签列表
labels = label_list
val_result, class_report = results_test(preds_val, out_label_ids, labels)
print("总体评估结果:", val_result)
print(class_report)

## 推理

您可以按如下方式加载模型进行推理：

In [ ]:
from mindnlp.transformers import LayoutLMv3Processor, LayoutLMv3ForTokenClassification
processor_infer = LayoutLMv3Processor.from_pretrained("./LayoutLMv3/processorPretrained", apply_ocr=False)  
model_infer = LayoutLMv3ForTokenClassification.from_pretrained(pretrained_model_path)

In [ ]:
from PIL import ImageDraw, ImageFont, Image
import numpy as np
from mindspore import Tensor

显示可视化结果

In [ ]:

def inference(image, words, boxes):
    """
    参数：
        image: 图像对象
        words：一个列表，待进行序列标注的文本token
        boxes: words列表的每个token在图像image的"检测框"的位置信息
    return：一个列表word_labels，元素对应 words的每个token的标注标签
    """
    # 打印边界框样例，用于调试
    print(f"模型输入边界框样例: {boxes[0]}")
    
    # 确保所有边界框都是归一化到0-1000范围的，处理器内部会有变换
    # 但我们需要确保输入时已经归一化到预期范围
    # 获取图像尺寸，考虑不同的图像格式
    if hasattr(image, 'size'):  # PIL图像
        width, height = image.size
    elif hasattr(image, 'shape'):  # NumPy数组
        if len(image.shape) == 3:  # 彩色图像 (高度, 宽度, 通道)
            height, width = image.shape[0], image.shape[1]
        else:  # 灰度图像或其他格式
            height, width = image.shape[0], image.shape[1]
    else:
        # 如果无法获取尺寸，尝试将图像转换为PIL图像
        try:
            from PIL import Image
            if not isinstance(image, Image.Image):
                image_pil = Image.fromarray(image)
                width, height = image_pil.size
            else:
                width, height = image.size
        except Exception as e:
            print(f"无法获取图像尺寸: {e}")
            # 如果无法确定尺寸并且边界框已经是归一化的，可以跳过归一化步骤
            if all(box[0] <= 1000 and box[1] <= 1000 and box[2] <= 1000 and box[3] <= 1000 for box in boxes):
                print("假设边界框已经归一化")
                width, height = 1000, 1000  # 用于调试
            else:
                raise ValueError("无法确定图像尺寸，无法归一化边界框")
    
    print(f"图像尺寸: 宽度={width}, 高度={height}")
    
    # 检查边界框格式，确保已经归一化
    first_box = boxes[0]
    if max(first_box) > 1000:
        # 如果边界框不是归一化的，进行归一化
        print("警告：输入边界框未归一化，正在进行归一化...")
        normalized_boxes = []
        for box in boxes:
            normalized_boxes.append([
                int(1000 * (box[0] / width)),
                int(1000 * (box[1] / height)),
                int(1000 * (box[2] / width)),
                int(1000 * (box[3] / height)),
            ])
        boxes = normalized_boxes
        print(f"归一化后边界框样例: {boxes[0]}")
    
    # 若加载 processor 时 apply_ocr=True，此时只需传输 图像数据即可，processor内部会对输入图像image进行OCR识别
    encoded_inputs = processor_infer(image, words, boxes=boxes,    # boxes 和 words应该分别是 OCR 的检测阶段和文本识别阶段得到的结果 
                                 word_labels = [-1] * len(words),  # 因为实际的token标签还未知（待模型预测出来），所以这里统一用 -1 代替
                                 padding='max_length', truncation=True, return_tensors="np")  # 这里不能直接通过设置return_tensors='ms'输出Tensor
    
    # 打印处理后的边界框样例，用于调试
    if 'bbox' in encoded_inputs:
        print(f"处理器输出边界框样例: {encoded_inputs['bbox'][0][0]}")
    
    # 对LayoutLMv3所需的输入进行转换
    input_ids = Tensor(encoded_inputs['input_ids'])
    attention_mask = Tensor(encoded_inputs['attention_mask'])
    bbox = Tensor(encoded_inputs['bbox'])
    pixel_values = Tensor(encoded_inputs['pixel_values'])  # v3使用pixel_values而非image
    
    model_infer.set_train(False)
    # forward pass
    outputs = model_infer(
        input_ids=input_ids, 
        attention_mask=attention_mask, 
        bbox=bbox, 
        pixel_values=pixel_values  # 使用pixel_values
    )
    
    # 模型预测的标签的id
    prediction_indices = outputs.logits.argmax(-1).squeeze().tolist()
    
    # 模型对每个token（包含特殊标记作用的token）的id都预测出了对应的标签id，注意后续取出实际预测的token标签时跳过特殊token（开始、结束、填充等标记）的标签
    labels = encoded_inputs['labels'].squeeze().tolist()
    label_special = labels[0]  # 特殊标记的标签，一般为-100
    
    # 获取预测的words每个token元素对应的文本标注
    prediction_labels = [id2label[label_pred_id] for gt, label_pred_id in zip(labels, prediction_indices) if gt != label_special]
    
    return prediction_labels

# 准备模拟OCR函数
def draw_rectangle(image, boxes: list = []):
    """
    绘制图像image的文本矩阵框，boxes的每个元素为[x1,y1,x3,y3]
    return 绘制了文本框的图像
    """
    # 深度拷贝
    copied_image_array = np.array(image)
    copied_image_deep = Image.fromarray(copied_image_array)
    draw = ImageDraw.Draw(copied_image_deep, "RGBA")   

    font = ImageFont.load_default()

    for box in boxes:
        # 确保边界框坐标为整数
        box_coords = [int(coord) for coord in box]
        # 使用draw.rectangle()方法在图片上绘制矩形框
        draw.rectangle(box_coords, outline='#FF0000', width=2)
    
    return copied_image_deep

# 从测试集中获取一个样本进行推理
sample_idx = 0  # 可以选择任意样本索引

# 获取原始图像、文本和边界框
image = eval_funsd_dataset.images[sample_idx]
words = eval_funsd_dataset.tokens[sample_idx]
boxes = eval_funsd_dataset.boxes[sample_idx]

# 将图像转换为PIL图像
image_pil = Image.fromarray(image)

# 如果需要，对边界框进行归一化处理
def normalize_bbox(bbox, width, height):
    """
    归一化边界框坐标到0-1000范围内
    注意：确保坐标顺序是 [x_min, y_min, x_max, y_max]
    """
    return [
        int(1000 * (bbox[0] / width)),
        int(1000 * (bbox[1] / height)),
        int(1000 * (bbox[2] / width)),
        int(1000 * (bbox[3] / height)),
    ]

# 检查边界框是否已经归一化（通常FUNSD数据集中的边界框已经归一化，但为了安全起见）
width, height = image_pil.size
first_box = boxes[0]
is_normalized = True
if max(first_box) > 1000:  # 如果有值大于1000，可能需要归一化
    normalized_boxes = [normalize_bbox(box, width, height) for box in boxes]
    is_normalized = False
else:
    normalized_boxes = boxes.copy()  # 使用复制以避免修改原始数据

# 计算反归一化的边界框，用于显示
denormalized_boxes = []
for box in normalized_boxes:
    denorm_box = [
        int(box[0] * width / 1000),
        int(box[1] * height / 1000),
        int(box[2] * width / 1000),
        int(box[3] * height / 1000)
    ]
    denormalized_boxes.append(denorm_box)

# 绘制反归一化边界框 - 使用从归一化边界框计算回来的边界框
img_with_rectangles = draw_rectangle(image_pil, denormalized_boxes)

# 尝试推理并捕获可能的错误
try:
    predicted_labels = inference(image, words, normalized_boxes)
    
    # 打印结果
    print("文本:", words)
    print("预测标签:", predicted_labels)
    print("边界框是否已归一化:", is_normalized)
    print("边界框例子:", boxes[0])
    print("归一化边界框例子:", normalized_boxes[0])
    print("反归一化边界框例子:", denormalized_boxes[0])
    
    # 打印标签统计
    from collections import Counter
    label_counts = Counter(predicted_labels)
    print("标签统计:", dict(label_counts))
    
    # 创建带标签的可视化图像 - 使用反归一化的边界框坐标
    visualization = visualize_predictions(image_pil, denormalized_boxes, words, predicted_labels)
except Exception as e:
    
    # 如果没有成功得到预测标签，创建一个空的标签列表
    predicted_labels = ["未知"] * len(words)
    visualization = visualize_predictions(image_pil, denormalized_boxes, words, predicted_labels)

# 显示图像（如果在支持显示的环境中）
img_with_rectangles.show()

# 可选：保存带标注的图像
output_dir = "./LayoutLMv3/results"
import os
os.makedirs(output_dir, exist_ok=True)
img_with_rectangles.save(f"{output_dir}/sample_{sample_idx}_prediction.png")

# 将预测标签可视化在图像上
def visualize_predictions(image, boxes, words, labels):
    """
    在图像上可视化文本和对应的标签
    
    Args:
        image: PIL图像
        boxes: 边界框坐标列表（原始像素坐标，非归一化）
        words: 文本列表
        labels: 标签列表
    
    Returns:
        带有标注的PIL图像
    """
    # 深度拷贝图像
    img_copy = image.copy()
    draw = ImageDraw.Draw(img_copy)
    
    # 尝试加载字体，如果失败则使用默认字体
    try:
        font = ImageFont.truetype("arial.ttf", 15)
    except IOError:
        font = ImageFont.load_default()
    
    # 颜色映射 - 为不同标签类型设置不同颜色
    color_map = {
        'O': '#000000',  # 黑色
        'B-HEADER': '#FF0000',  # 红色
        'I-HEADER': '#FF3333',  # 浅红色
        'B-QUESTION': '#0000FF',  # 蓝色
        'I-QUESTION': '#3333FF',  # 浅蓝色
        'B-ANSWER': '#00FF00',  # 绿色
        'I-ANSWER': '#33FF33',  # 浅绿色
    }
    
    # 绘制边界框和标签
    for box, word, label in zip(boxes, words, labels):
        # 确保边界框坐标为整数
        box_coords = [int(coord) for coord in box]
        
        # 绘制边界框
        color = color_map.get(label, '#000000')
        draw.rectangle(box_coords, outline=color, width=2)
        
        # 绘制标签和文本
        text = f"{word} ({label})"
        text_position = (box_coords[0], max(0, box_coords[1] - 15))  # 在边界框上方显示，但不超出图像
        
        # 绘制背景矩形以增强文本可读性
        text_bbox = draw.textbbox(text_position, text, font=font)
        draw.rectangle(text_bbox, fill='white')
        
        # 绘制文本
        draw.text(text_position, text, fill=color, font=font)
    
    return img_copy

# 创建带标签的可视化图像 - 使用反归一化的边界框坐标
visualization = visualize_predictions(image_pil, denormalized_boxes, words, predicted_labels)

# 保存可视化结果
visualization.save(f"{output_dir}/sample_{sample_idx}_visualization.png")

# 显示可视化结果（如果在支持显示的环境中）
visualization.show()


让我们只比较标签不是-100的位置的预测和标签。我们还希望有这些（非标准化）的边界框：

In [ ]:
def unnormalize_box(bbox, width, height):
    """
    将归一化的边界框转换回原始坐标
    
    Args:
        bbox: 归一化的边界框坐标 [x_min, y_min, x_max, y_max]
        width: 图像宽度
        height: 图像高度
        
    Returns:
        反归一化的边界框坐标
    """
    return [
        width * (bbox[0] / 1000),
        height * (bbox[1] / 1000),
        width * (bbox[2] / 1000),
        height * (bbox[3] / 1000),
    ]

def iob_to_label(label):
    """
    将IOB格式的标签转换为简单标签
    
    Args:
        label: IOB格式的标签，如'B-HEADER'或'O'
        
    Returns:
        简单标签，如'header'或'other'
    """
    # 处理'O'标签
    if label == 'O':
        return 'other'
    
    # 处理B-和I-格式的标签
    if label.startswith('B-') or label.startswith('I-'):
        label = label[2:]  # 去掉B-或I-前缀
    
    if not label:
        return 'other'
    return label.lower()

def visualize_advanced(image, boxes, predictions, labels=None):
    """
    在图像上可视化文本和预测标签，更高级的版本
    
    Args:
        image: PIL图像
        boxes: 归一化的边界框坐标列表
        predictions: 预测的标签列表
        labels: 真实标签列表，如果有的话
        
    Returns:
        带有标注的PIL图像
    """
    img_copy = image.copy()
    draw = ImageDraw.Draw(img_copy)
    
    width, height = image.size
    
    # 定义标签到颜色的映射
    label2color = {
        'question': '#0000FF',  # 蓝色
        'answer': '#00FF00',    # 绿色
        'header': '#FFA500',    # 橙色
        'other': '#8A2BE2'      # 紫色
    }
    
    # 尝试加载字体
    try:
        font = ImageFont.truetype("arial.ttf", 15)
    except IOError:
        font = ImageFont.load_default()
    
    # 反归一化边界框并绘制标签
    for i, (box, prediction) in enumerate(zip(boxes, predictions)):
        # 反归一化边界框
        denorm_box = unnormalize_box(box, width, height)
        denorm_box = [int(coord) for coord in denorm_box]  # 转换为整数
        
        # 获取简化后的标签和对应的颜色
        predicted_label = iob_to_label(prediction)
        color = label2color.get(predicted_label, '#000000')
        
        # 绘制边界框
        draw.rectangle(denorm_box, outline=color, width=2)
        
        # 在边界框上方显示标签
        text_position = (denorm_box[0] + 10, max(0, denorm_box[1] - 15))
        
        # 绘制背景矩形以增强文本可读性
        text_bbox = draw.textbbox(text_position, predicted_label, font=font)
        draw.rectangle(text_bbox, fill='white')
        
        # 绘制标签文本
        draw.text(text_position, predicted_label, fill=color, font=font)
    
    return img_copy

# 尝试使用新的可视化函数
try:
    # 获取需要的数据
    encoded_inputs = processor_infer(image, words, boxes=normalized_boxes, 
                                    word_labels = [-1] * len(words),
                                    padding='max_length', truncation=True, return_tensors="np")
    
    # 获取token级别的边界框
    token_boxes = encoded_inputs['bbox'].squeeze().tolist()
    
    # 获取对应的标签（-100是填充标签）
    labels = encoded_inputs['labels'].squeeze().tolist()
    
    # 处理预测结果
    model_infer.set_train(False)
    input_ids = Tensor(encoded_inputs['input_ids'])
    attention_mask = Tensor(encoded_inputs['attention_mask'])
    bbox = Tensor(encoded_inputs['bbox'])
    pixel_values = Tensor(encoded_inputs['pixel_values'])
    
    outputs = model_infer(
        input_ids=input_ids, 
        attention_mask=attention_mask, 
        bbox=bbox, 
        pixel_values=pixel_values
    )
    
    # 获取预测结果
    predictions = outputs.logits.argmax(-1).squeeze().tolist()
    
    # 过滤掉填充标签
    true_predictions = [id2label[pred] for pred, label in zip(predictions, labels) if label != -100]
    true_boxes = [box for box, label in zip(token_boxes, labels) if label != -100]
    
    # 创建高级可视化
    advanced_viz = visualize_advanced(image_pil, true_boxes, true_predictions)
    
    # 保存和显示结果
    advanced_viz.save(f"{output_dir}/sample_{sample_idx}_advanced_viz.png")
    advanced_viz.show()
    
    print(f"高级可视化已保存到 {output_dir}/sample_{sample_idx}_advanced_viz.png")
    
except Exception as e:
    print(f"高级可视化过程中出现错误: {e}")




将此与真实标签进行比较：

In [ ]:

# 添加比较功能: 比较预测结果与真实标签
def compare_predictions_with_ground_truth(image, example, id2label, width=None, height=None):
    """
    在图像上可视化预测结果与真实标签的比较
    
    Args:
        image: PIL图像
        example: 数据样本，包含tokens、bboxes和ner_tags
        id2label: 标签id到标签文本的映射
        width: 图像宽度，如果为None则从图像获取
        height: 图像高度，如果为None则从图像获取
    
    Returns:
        带有真实标签和预测标签的PIL图像
    """
    if width is None or height is None:
        width, height = image.size
    
    # 创建两个图像副本
    img_ground_truth = image.copy()
    draw_gt = ImageDraw.Draw(img_ground_truth)
    
    # 定义标签到颜色的映射
    label2color = {
        'question': '#0000FF',  # 蓝色
        'answer': '#00FF00',    # 绿色
        'header': '#FFA500',    # 橙色
        'other': '#8A2BE2'      # 紫色
    }
    
    # 尝试加载字体
    try:
        font = ImageFont.truetype("arial.ttf", 15)
    except IOError:
        font = ImageFont.load_default()
    
    # 绘制真实标签
    for word, box, label_id in zip(example['tokens'], example['bboxes'], example['ner_tags']):
        # 反归一化边界框
        if max(box) <= 1000:  # 如果边界框已经归一化
            box = unnormalize_box(box, width, height)
        
        # 转换为整数坐标
        box = [int(coord) for coord in box]
        
        # 获取真实标签
        if isinstance(label_id, (int, float)):
            label_id = int(label_id)
            actual_label = iob_to_label(id2label[label_id])
        else:
            # 如果label_id已经是字符串
            actual_label = iob_to_label(label_id)
        
        # 获取颜色
        color = label2color.get(actual_label, '#000000')
        
        # 绘制边界框
        draw_gt.rectangle(box, outline=color, width=2)
        
        # 绘制标签
        text_position = (box[0] + 10, max(0, box[1] - 15))
        text_bbox = draw_gt.textbbox(text_position, actual_label, font=font)
        draw_gt.rectangle(text_bbox, fill='white')
        draw_gt.text(text_position, actual_label, fill=color, font=font)
    
    return img_ground_truth

# 尝试获取并显示真实标签
try:
    # 获取测试样本 - 这里直接使用test_dataset而不是test_dataset["test"]
    test_example = next(test_dataset.create_dict_iterator())
    sample_idx = 0
    
    # 获取测试样本中的图像和标签 - 直接使用我们已经准备好的数据
    example = {
        'image': eval_funsd_dataset.images[sample_idx],
        'tokens': eval_funsd_dataset.tokens[sample_idx],
        'bboxes': eval_funsd_dataset.boxes[sample_idx],
        'ner_tags': eval_funsd_dataset.labels[sample_idx]
    }
    
    # 将图像转换为PIL格式
    image_pil = Image.fromarray(example['image']).convert("RGB")
    width, height = image_pil.size
    
    # 创建真实标签可视化
    ground_truth_viz = compare_predictions_with_ground_truth(
        image_pil, 
        example, 
        id2label, 
        width, 
        height
    )
    
    # 保存真实标签可视化
    ground_truth_viz.save(f"{output_dir}/sample_{sample_idx}_ground_truth.png")
    ground_truth_viz.show()
    
except Exception as e:
    print(f"创建真实标签可视化时出错: {e}")
    import traceback
    traceback.print_exc()




## 注意：没有标签时的推理

上面的代码使用“标签”来确定哪些标记是否位于特定单词的开头。当然，在推理时，您无法访问任何标签。在这种情况下，您可以利用标记器返回的`offset_mapping`。